## 1 — Imports

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType,
    BooleanType, TimestampType, DoubleType, FloatType
)
from pyspark.sql.functions import current_timestamp, col, lit
from datetime import datetime, timezone
import uuid, json

## 2 — Job Parameters (Widget Declarations)
> **Must be the first code executed.** Databricks injects Job parameter values into widgets before the notebook runs. Declare before any `.get()` call.

In [0]:
# ── Job Parameters ────────────────────────────────────────────────────────────
# Default values are used during interactive runs.
# Databricks Job overrides these at runtime via the Parameters section.
# Key names here must match exactly what the Job defines.

dbutils.widgets.text(
    "catalog_name", "T20_catalog_dev_CopyInto",
    "Catalog Name"
)
dbutils.widgets.text(
    "schema_name", "bronze",
    "Schema Name"
)
dbutils.widgets.text(
    "source_path",
    "abfss://cricinfo-mens-international@adlschitturidemo.dfs.core.windows.net/",
    "Source Base Path"
)
dbutils.widgets.text(
    "force_reload", "false",
    "Force Reload — true re-ingests all files (use after TRUNCATE)"
)
dbutils.widgets.dropdown(
    "table_to_load", "all",
    ["all", "match_events", "match_metadata", "match_players"],
    "Table To Load"
)
dbutils.widgets.text(
    "DQ_notebook_path_base_Path",  "/Workspace/Users/pradeepchitturi@gmail.com/Dev_T20_CommentaryParser_CopyInto", "DQ Notebook Base Path"
)

## 3 — Capture Parameter Values

In [0]:
# ── Capture Job Parameters ────────────────────────────────────────────────────
CATALOG_NAME  = dbutils.widgets.get("catalog_name")
SCHEMA_NAME   = dbutils.widgets.get("schema_name")
FULL_SCHEMA   = f"{CATALOG_NAME}.{SCHEMA_NAME}"
SOURCE_PATH   = dbutils.widgets.get("source_path")
FORCE_RELOAD  = dbutils.widgets.get("force_reload").strip().lower() == "true"
TABLE_TO_LOAD = dbutils.widgets.get("table_to_load").strip()

print("=" * 65)
print("JOB PARAMETERS RECEIVED")
print("=" * 65)
print(f"  Catalog      : {CATALOG_NAME}")
print(f"  Schema       : {SCHEMA_NAME}")
print(f"  Full Schema  : {FULL_SCHEMA}")
print(f"  Source Path  : {SOURCE_PATH}")
print(f"  Force Reload : {FORCE_RELOAD}")
print(f"  Table        : {TABLE_TO_LOAD}")
print("=" * 65)

## 4 — StructType Schema Definitions
> Single source of truth. `CREATE TABLE` DDL, `COPY INTO` SELECT casts, and `schemaHints` are all auto-generated from these definitions.

In [0]:
# ── StructType Schema Definitions ─────────────────────────────────────────────
# Single source of truth for every table.
# CREATE TABLE DDL, COPY INTO SELECT casts, and schemaHints
# are ALL derived from these definitions — no manual SQL needed.

# ── match_events ──────────────────────────────────────────────────────────────
match_events_schema = StructType([
    StructField("Batchid",            StringType(),    True),
    StructField("match_ball_number",  LongType(),      True),
    StructField("ball",               StringType(),    True),
    StructField("event",              StringType(),    True),
    StructField("commentary",         StringType(),    True),
    StructField("bowler",             StringType(),    True),
    StructField("batsman",            StringType(),    True),
    StructField("innings",            StringType(),    True),
    StructField("matchid",            LongType(),      True),
    StructField("is_super_over",      BooleanType(),   True),
    StructField("load_timestamp",     TimestampType(), True),
    StructField("source_file",        StringType(),    True)
])

# ── match_metadata ────────────────────────────────────────────────────────────
match_metadata_schema = StructType([
    StructField("Batchid",                  StringType(),    True),
    StructField("ground",                   StringType(),    True),
    StructField("toss",                     StringType(),    True),
    StructField("series",                   StringType(),    True),
    StructField("season",                   StringType(),    True),
    StructField("player_of_the_match",      StringType(),    True),
    StructField("player_of_the_series",     StringType(),    True),
    StructField("hours_of_play_local_time", StringType(),    True),
    StructField("match_days",               StringType(),    True),
    StructField("t20_debut",                StringType(),    True),
    StructField("t20i_debut",               StringType(),    True),
    StructField("umpires",                  StringType(),    True),
    StructField("tv_umpire",                StringType(),    True),
    StructField("reserve_umpire",           StringType(),    True),
    StructField("match_referee",            StringType(),    True),
    StructField("points",                   StringType(),    True),
    StructField("match_number",             StringType(),    True),
    StructField("matchid",                  LongType(),      True),
    StructField("player_replacements",      StringType(),    True),
    StructField("first_innings",            StringType(),    True),
    StructField("second_innings",           StringType(),    True),
    StructField("has_super_over",           BooleanType(),   True),
    StructField("super_over_count",         IntegerType(),   True),
    StructField("series_result",            StringType(),    True),
    StructField("load_timestamp",           TimestampType(), True),
    StructField("source_file",              StringType(),    True)
])

# ── match_players ─────────────────────────────────────────────────────────────
match_players_schema = StructType([
    StructField("Batchid",          StringType(),    True),
    StructField("matchid",          LongType(),      True),
    StructField("innings",          StringType(),    True),
    StructField("team",             StringType(),    True),
    StructField("player_name",      StringType(),    True),
    StructField("batted",           BooleanType(),   True),
    StructField("batting_position", FloatType(),   True),
    StructField("player_type",      StringType(),    True),
    StructField("retired",          StringType(),    True),
    StructField("not_out",          StringType(),    True),
    StructField("bowled",           StringType(),    True),
    StructField("load_timestamp",   TimestampType(), True),
    StructField("source_file",      StringType(),    True)
])

## 5 — Schema Utility Functions
> `schema_to_ddl` · `schema_to_select` · `schema_to_hints`

In [0]:
# ── Schema Utility Functions ──────────────────────────────────────────────────
# Derive DDL, SELECT casts, and schemaHints from a StructType.
# Adding a new column = update StructType only. Nothing else changes.

# Columns injected by the pipeline — NOT read from source files.
# Excluded from schemaHints and cast list; added as SQL expressions.
PIPELINE_COLUMNS = {"load_timestamp", "source_file", "Batchid"}


def spark_type_to_sql(spark_type) -> str:
    """Map a Spark DataType to its SQL DDL string."""
    mapping = {
        StringType():    "STRING",
        LongType():      "BIGINT",
        IntegerType():   "INT",
        BooleanType():   "BOOLEAN",
        TimestampType(): "TIMESTAMP",
        DoubleType():    "DOUBLE",
        FloatType():     "FLOAT",
    }
    return mapping.get(spark_type, "STRING")


def schema_to_ddl(schema: StructType) -> str:
    """
    StructType → CREATE TABLE column definitions.

    Example output (per field):
        match_ball_number  BIGINT,
        is_super_over      BOOLEAN,
        load_timestamp     TIMESTAMP
    """
    lines = []
    for field in schema.fields:
        sql_type = spark_type_to_sql(field.dataType)
        lines.append(f"    {field.name} {sql_type}")
    return ",\n".join(lines)


def schema_to_select(schema: StructType, batch_id: str) -> str:
    """
    StructType → SELECT clause for COPY INTO.

    Every source column gets an explicit CAST to its declared type,
    eliminating all type-inference mismatches.
    Pipeline columns are injected as SQL expressions.

    Example output (per field):
        CAST(match_ball_number AS BIGINT)   AS match_ball_number,
        CAST(is_super_over     AS BOOLEAN)  AS is_super_over,
        _metadata.file_path                 AS source_file,
        current_timestamp()                 AS load_timestamp,
        'batch_xyz'                         AS Batchid
    """
    source_cols   = []
    pipeline_cols = []

    for field in schema.fields:
        name     = field.name
        sql_type = spark_type_to_sql(field.dataType)

        if name == "load_timestamp":
            pipeline_cols.append(
                "current_timestamp()                AS load_timestamp"
            )
        elif name == "source_file":
            pipeline_cols.append(
                "_metadata.file_path                AS source_file"
            )
        elif name == "Batchid":
            pipeline_cols.append(
                f"\'{batch_id}\'                     AS Batchid"
            )
        else:
            source_cols.append(
                f"CAST({name:<28} AS {sql_type:<10}) AS {name}"
            )

    all_cols = source_cols + pipeline_cols
    return ",\n                ".join(all_cols)


def schema_to_hints(schema: StructType) -> str:
    """
    StructType → schemaHints string for FORMAT_OPTIONS.

    Only non-STRING, non-pipeline columns are listed —
    Spark defaults to STRING for ambiguous values so those
    need no override. Hints are the first line of defence;
    CAST in SELECT is the final guarantee.

    Example output:
        'match_ball_number BIGINT, matchid BIGINT, is_super_over BOOLEAN'
    """
    hints = []
    for field in schema.fields:
        if field.name in PIPELINE_COLUMNS:
            continue
        sql_type = spark_type_to_sql(field.dataType)
        if sql_type != "STRING":
            hints.append(f"{field.name} {sql_type}")
    return ", ".join(hints)


# ── Quick sanity check ────────────────────────────────────────────────────────
print("Schema utilities loaded.")
print(f"  match_events  hints : {schema_to_hints(match_events_schema)}")
print(f"  match_metadata hints: {schema_to_hints(match_metadata_schema)}")
print(f"  match_players hints : {schema_to_hints(match_players_schema)}")

## 6 — Pipeline Class — `IPLDataPipelineCopyInto`

In [0]:
class IPLDataPipelineCopyInto:
    """
    Bronze ingestion pipeline — COPY INTO edition.

    Design principles
    -----------------
    * Schema-driven  : StructType is the single source of truth.
                       DDL, SELECT casts, and schemaHints are all derived
                       automatically — no manual SQL duplication.
    * Idempotent     : COPY INTO tracks loaded files in the Delta transaction
                       log (force=false). Re-running never creates duplicates.
    * Parameterised  : All env values come from dbutils.widgets (Job params).
    * Type-safe      : Three layers of type enforcement:
                         1. schemaHints   — how Spark reads the file
                         2. CAST in SELECT — before writing to Delta
                         3. Delta schema   — enforced at write time
    """

    def __init__(self, source_base_path, catalog_name, schema_name):
        self.source_base_path = source_base_path.rstrip("/") + "/"
        self.catalog_name     = catalog_name
        self.schema_name      = schema_name
        self.full_schema      = f"{catalog_name}.{schema_name}"
        self.batch_id         = self._generate_batch_id()

        self.tables = {
            "match_events": {
                "pattern":     "match_events_data.csv",
                "format":      "csv",
                "table_name":  f"{self.full_schema}.match_events",
                "description": "Ball-by-ball match events",
                "schema":      match_events_schema,
                "zorder_cols": ["bowler", "batsman"]
            },
            "match_metadata": {
                "pattern":     "metadata_data.json",
                "format":      "json",
                "table_name":  f"{self.full_schema}.match_metadata",
                "description": "Match metadata and results",
                "schema":      match_metadata_schema,
                "zorder_cols": []
            },
            "match_players": {
                "pattern":     "match_players_data.csv",
                "format":      "csv",
                "table_name":  f"{self.full_schema}.match_players",
                "description": "Player information per match",
                "schema":      match_players_schema,
                "zorder_cols": ["player_name", "team"]
            }
        }

    # ── Batch ID ──────────────────────────────────────────────────────────────
    @staticmethod
    def _generate_batch_id() -> str:
        """
        Use Databricks job/run context when available, else fall back to UUID.
        Stored in every row as Batchid for lineage and debugging.
        """
        try:
            ctx    = (dbutils.notebook.entry_point
                             .getDbutils().notebook().getContext())
            job_id = ctx.jobId().get()
            run_id = ctx.idInJob().get()
            return f"job_{job_id}_run_{run_id}"
        except Exception:
            ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            return f"interactive_{ts}_{uuid.uuid4().hex[:8]}"

    # ── Create Tables ─────────────────────────────────────────────────────────
    def create_tables(self):
        """
        Create all Delta tables using DDL auto-generated from StructType.
        Safe to run repeatedly — CREATE TABLE IF NOT EXISTS is idempotent.
        """
        print("\n" + "=" * 65)
        print("CREATE TABLES (schema derived from StructType)")
        print("=" * 65)

        for table_name, config in self.tables.items():
            table_full_name = config["table_name"]
            col_definitions = schema_to_ddl(config["schema"])

            ddl = f"""
                CREATE TABLE IF NOT EXISTS {table_full_name} (
                {col_definitions}
                )
                USING DELTA
                PARTITIONED BY (matchid)
                COMMENT '{config["description"]}'
                TBLPROPERTIES (
                    'delta.enableChangeDataFeed' = 'true',
                    'pipeline.layer'             = 'bronze',
                    'pipeline.source'            = 'copy_into',
                    'pipeline.schema_version'    = '1.0'
                )
            """

            print(f"\nCreating {table_full_name}...")
            spark.sql(ddl)
            print(f"  ✓ Ready")

        print("\n✓ All tables created successfully")

    # ── Load Single Table ─────────────────────────────────────────────────────
    def load_table(self, table_name: str, force: bool = False):
        """
        Load one table using COPY INTO.

        File tracking
        -------------
        COPY INTO records every loaded file path in the Delta transaction log
        under operationParameters.  On subsequent runs with force=False, Spark
        reads the log, builds the already-seen path list, and skips those files.
        Only genuinely new files are ingested.

        Type safety layers
        ------------------
        1. inferSchema=false   → all raw columns arrive as STRING
        2. schemaHints         → Spark coerces specific columns while reading
        3. CAST in SELECT      → final explicit cast before write to Delta
        4. Delta schema        → write-time enforcement

        Args
        ----
        table_name : Key from self.tables
        force      : True = reload ALL files (bypasses transaction log check).
                     WARNING — will duplicate rows unless table is TRUNCATEd first.
        """
        config          = self.tables[table_name]
        schema          = config["schema"]
        table_full_name = config["table_name"]
        file_format     = config["format"].upper()
        force_flag      = "true" if force else "false"

        # Glob matches: cricket_commentary/matchid=<id>/<filename>
        # Single-level * is fully supported by COPY INTO
        # ** recursive glob is NOT supported — do not use it
        source_glob = (
            f"{self.source_base_path}"
            f"cricket_commentary/matchid=*/{config['pattern']}"
        )

        # Derive SELECT clause and hints from StructType
        select_clause = schema_to_select(schema, self.batch_id)
        schema_hints  = schema_to_hints(schema)

        # Format-specific options
        # inferSchema=false ensures every column is read as STRING first,
        # then schemaHints + CAST handle typing — removes all inference risk.
        if file_format == "CSV":
            format_options = f"""
                FORMAT_OPTIONS (
                    'header'      = 'true',
                    'inferSchema' = 'false',
                    'schemaHints' = '{schema_hints}'
                )"""
        else:   # JSON
            format_options = f"""
                FORMAT_OPTIONS (
                    'inferSchema' = 'false',
                    'schemaHints' = '{schema_hints}'
                )"""

        copy_into_sql = f"""
            COPY INTO {table_full_name}
            FROM (
                SELECT
                {select_clause}
                FROM '{source_glob}'
            )
            FILEFORMAT = {file_format}
            {format_options}
            COPY_OPTIONS (
                'mergeSchema' = 'true',
                'force'       = '{force_flag}'
            )
        """

        print(f"\n{'=' * 65}")
        print(f"Table        : {table_full_name}")
        print(f"Format       : {file_format}")
        print(f"Source glob  : {source_glob}")
        print(f"Schema hints : {schema_hints}")
        print(f"Force reload : {force_flag}")
        print(f"Batch ID     : {self.batch_id}")
        print(f"{'=' * 65}")
        print("Executing COPY INTO...")

        result_df = spark.sql(copy_into_sql)

        # COPY INTO returns a result row:
        # num_affected_rows | num_inserted_rows | num_skipped_corrupt_files
        result_df.show(truncate=False)

        # Post-load statistics
        count      = spark.table(table_full_name).count()
        partitions = spark.table(table_full_name).select("matchid").distinct().count()

        print(f"  ✓ COPY INTO completed")
        print(f"  ✓ Total rows in table  : {count:,}")
        print(f"  ✓ Distinct matchids    : {partitions}")

    # ── Ingest All Tables ─────────────────────────────────────────────────────
    def ingestion_copy_into(self, force: bool = False) -> dict:
        """
        Run COPY INTO for all bronze tables sequentially.

        Args
        ----
        force : Passed through to load_table — see load_table docstring.

        Returns
        -------
        dict : {table_name: "SUCCESS" | "FAILED: <reason>"}
        """
        print("\n" + "=" * 65)
        print("BATCH LOAD — ALL TABLES (COPY INTO)")
        print("=" * 65)

        results = {}
        for table_name in self.tables:
            try:
                self.load_table(table_name, force=force)
                results[table_name] = "SUCCESS"
            except Exception as e:
                import traceback
                traceback.print_exc()
                results[table_name] = f"FAILED: {e}"

        # Summary
        print("\n" + "=" * 65)
        print("COPY INTO LOAD SUMMARY")
        print("=" * 65)
        for table_name, status in results.items():
            icon = "✓" if status == "SUCCESS" else "✗"
            print(f"  {icon} {self.tables[table_name]['table_name']}: {status}")

        return results

    # ── Force Reload ──────────────────────────────────────────────────────────
    def force_reload(self, table_name: str):
        """
        Re-ingest ALL files for one table regardless of prior loads.

        Equivalent of deleting the Auto Loader checkpoint.
        COPY INTO equivalent: COPY_OPTIONS ('force' = 'true').

        IMPORTANT — Always TRUNCATE the table first:
            spark.sql(f"TRUNCATE TABLE {FULL_SCHEMA}.{table_name}")
            pipeline.force_reload("{table_name}")
        Skipping TRUNCATE will create duplicate rows.
        """
        table_full_name = self.tables[table_name]["table_name"]
        print(f"⚠️  WARNING: Force-reloading {table_full_name}")
        print(f"   All files will be re-ingested.")
        print(f"   Ensure you have TRUNCATEd the table to avoid duplicates.")
        self.load_table(table_name, force=True)

    # ── Optimize ──────────────────────────────────────────────────────────────
    def optimize_table(self, table_name: str):
        """
        OPTIMIZE the Delta table.
        Z-ORDER columns are defined per table in self.tables config.
        Z-ORDER must not be applied to the partition column (matchid).
        """
        config          = self.tables[table_name]
        table_full_name = config["table_name"]
        zorder_cols     = config.get("zorder_cols", [])

        print(f"\nOptimizing {table_full_name}...")

        if zorder_cols:
            cols = ", ".join(zorder_cols)
            print(f"  Z-ORDER BY: {cols}")
            spark.sql(f"OPTIMIZE {table_full_name} ZORDER BY ({cols})")
        else:
            spark.sql(f"OPTIMIZE {table_full_name}")

        print(f"  ✓ Optimization complete")

    def optimize_all(self):
        """Optimize all bronze tables."""
        print("\n" + "=" * 65)
        print("OPTIMIZING ALL TABLES")
        print("=" * 65)
        for table_name in self.tables:
            self.optimize_table(table_name)

    # ── Statistics ────────────────────────────────────────────────────────────
    def show_statistics(self):
        """Show row counts and sample data for all tables."""
        print("\n" + "=" * 65)
        print("TABLE STATISTICS")
        print("=" * 65)

        for table_name, config in self.tables.items():
            table_full_name = config["table_name"]
            try:
                df         = spark.table(table_full_name)
                count      = df.count()
                partitions = df.select("matchid").distinct().count()

                print(f"\n{table_full_name}")
                print(f"  Description : {config['description']}")
                print(f"  Format      : {config['format'].upper()}")
                print(f"  Total rows  : {count:,}")
                print(f"  Matchids    : {partitions}")
                display(df.limit(3))
            except Exception as e:
                print(f"  ✗ Error reading {table_full_name}: {e}")

    # ── Schema Inspector ──────────────────────────────────────────────────────
    def show_schemas(self):
        """
        Print the generated DDL, SELECT clause, and schema hints
        for every table — useful for debugging and documentation.
        """
        for table_name, config in self.tables.items():
            schema = config["schema"]
            print(f"\n{'=' * 65}")
            print(f"TABLE: {config['table_name']}")
            print(f"{'=' * 65}")
            print("\n── DDL (auto-generated) ──────────────────────────────")
            print(schema_to_ddl(schema))
            print("\n── Schema Hints ───────────────────────────────────────")
            print(schema_to_hints(schema))
            print("\n── SELECT Clause (auto-generated) ────────────────────")
            print(schema_to_select(schema, "<batch_id>"))
            print()

## 7 — Pipeline Execution

In [0]:
# ── Initialise Pipeline ───────────────────────────────────────────────────────
pipeline = IPLDataPipelineCopyInto(
    source_base_path = SOURCE_PATH,
    catalog_name     = CATALOG_NAME,
    schema_name      = SCHEMA_NAME
)

print(f"Pipeline initialised  — Batch ID: {pipeline.batch_id}")

In [0]:
# ── Create Bronze Tables ──────────────────────────────────────────────────────
# DDL is auto-generated from StructType — no manual SQL needed.
# Safe to re-run — CREATE TABLE IF NOT EXISTS is idempotent.

pipeline.create_tables()

In [0]:
# ── Inspect Auto-Generated SQL ────────────────────────────────────────────────
# Optional — shows what DDL, SELECT clause, and schema hints
# will be generated for each table before actually running COPY INTO.
# Useful for debugging schema issues.

pipeline.show_schemas()

In [0]:
# ── Execute COPY INTO ─────────────────────────────────────────────────────────
# force=False (default) → idempotent: already-loaded files are skipped
#                          Delta transaction log tracks loaded file paths
# force=True            → reload ALL files (use after TRUNCATE for reprocessing)

print("🚀 Running Batch LOAD with COPY INTO...")

if TABLE_TO_LOAD == "all":
    # Load all three tables
    results = pipeline.ingestion_copy_into(force=FORCE_RELOAD)
else:
    # Job passed a specific table name → load only that one
    pipeline.load_table(TABLE_TO_LOAD, force=FORCE_RELOAD)
    results = {TABLE_TO_LOAD: "SUCCESS"}

## 8 — Statistics & Validation

In [0]:
# ── Table Statistics ──────────────────────────────────────────────────────────
pipeline.show_statistics()

In [0]:
# ── Verify Tables in Catalog ──────────────────────────────────────────────────
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

In [0]:
# ── Sample Queries ────────────────────────────────────────────────────────────

# Row counts per match
display(spark.sql(f"""
    SELECT matchid, COUNT(*) AS row_count
    FROM   {FULL_SCHEMA}.match_events
    GROUP  BY matchid
    ORDER  BY matchid
"""))

# Latest batch loaded
display(spark.sql(f"""
    SELECT Batchid, load_timestamp, COUNT(*) AS rows
    FROM   {FULL_SCHEMA}.match_events
    GROUP  BY Batchid, load_timestamp
    ORDER  BY load_timestamp DESC
    LIMIT  5
"""))

## 9 — Optimize (Run After Large Loads)

In [0]:
# ── Optimize Tables ───────────────────────────────────────────────────────────
# Run after initial load or large incremental loads.
# Compacts small files and applies Z-ORDER for faster queries.
# Z-ORDER columns defined per table in pipeline.tables config.

# pipeline.optimize_all()               # all tables
# pipeline.optimize_table("match_events")  # single table

## 10 — Force Reload (Use Carefully — Replaces Auto Loader Checkpoint Reset)

In [0]:
# ── Force Reload ──────────────────────────────────────────────────────────────
# Equivalent of deleting the Auto Loader checkpoint.
# Use when you need to fully reprocess a table after a bug fix.
#
# ALWAYS truncate the table first to prevent duplicate rows:
#
#   spark.sql(f"TRUNCATE TABLE {FULL_SCHEMA}.match_events")
#   pipeline.force_reload("match_events")
#
#   spark.sql(f"TRUNCATE TABLE {FULL_SCHEMA}.match_metadata")
#   pipeline.force_reload("match_metadata")
#
#   spark.sql(f"TRUNCATE TABLE {FULL_SCHEMA}.match_players")
#   pipeline.force_reload("match_players")
#
# Or reload all at once:
#   for t in ["match_events", "match_metadata", "match_players"]:
#       spark.sql(f"TRUNCATE TABLE {FULL_SCHEMA}.{t}")
#   pipeline.ingestion_copy_into(force=True)

## 11 — Data Quality Checks

In [0]:
# ── Data Quality Checks ───────────────────────────────────────────────────────
# Run DQ notebook after ingestion completes.
# Passes catalog/schema so the DQ notebook knows which tables to validate.

# ✅ Uses notebook_path from job task parameter
NOTEBOOK_BASE_PATH = dbutils.widgets.get("DQ_notebook_path_base_Path")

dq_result = dbutils.notebook.run(
    f"{NOTEBOOK_BASE_PATH}/DataQualityRulesBronzeLayer",
    timeout_seconds = 600,
    arguments = {
        "catalog_name": CATALOG_NAME,
        "schema_name":  SCHEMA_NAME
    }
)
print(f"DQ Result: {dq_result}")

## 12 — Notebook Exit (Job Result)

In [0]:
# ── Notebook Exit ─────────────────────────────────────────────────────────────
# Returns a JSON summary to the caller (Databricks Job or parent notebook).
# dbutils.notebook.run() receives this string as its return value.

exit_payload = json.dumps({
    "status":        "SUCCESS" if all(v == "SUCCESS" for v in results.values()) else "PARTIAL",
    "results":       results,
    "batch_id":      pipeline.batch_id,
    "catalog":       CATALOG_NAME,
    "schema":        SCHEMA_NAME,
    "table":         TABLE_TO_LOAD,
    "force_reload":  FORCE_RELOAD
})

print(f"Exit payload: {exit_payload}")
dbutils.notebook.exit(exit_payload)